Weeky Assessment -Agentic Ai and RAG

🔷 Question 1: Build an End-to-End RAG + Agent System (25 Marks)
🧩 Scenario
You are an AI intern at an ed-tech company. Students frequently ask questions about:

Course policies (refunds, deadlines)
Lecture content (PDF notes)
Assignment deadlines
Their enrollment status
The company wants a single intelligent assistant that:

Answers questions using internal documents (PDFs, FAQs)
Fetches student-specific data (like enrollment or progress) using tools/APIs
Avoids hallucination and gives reliable answers
💻 Task
Design and implement a working prototype (pseudo-code or real code) for this system.

Your solution must include:

✅ 1. RAG Pipeline
How you will ingest and preprocess documents
Chunking strategy (with justification)
Embedding + vector store choice
Retrieval logic
How context is injected into the LLM
✅ 2. Agent Integration
Design an agent that decides:
When to use RAG
When to call a tool (e.g., get_student_status(student_id))
Show how tools are defined and used
✅ 3. End-to-End Flow
Write code or structured pseudo-code showing:

Input query
Retrieval step
Tool calling (if needed)
Final answer generation
✅ 4. Reliability Improvements
Show at least 2 techniques in code/design to:

Reduce hallucination
Improve answer quality
🎯 Expected Outcome
A working architecture/code that demonstrates:

RAG + Agent working together
Decision-making capability
Real-world practicality


In [15]:
# SETUP GROQ API KEY
import os

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('groq_api_key')
except:
    GROQ_API_KEY = os.getenv("GROQ_API_KEY")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY


# INSTALL DEPENDENCIES
!pip install -qqq langchain_text_splitters langchain_community langchain_core langchain_groq faiss-cpu sentence-transformers


# IMPORTS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.documents import Document


# LLM
llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",  # Updated Groq model as per user request
    temperature=0
)

# DOCUMENT LOADING
def load_documents():
    docs = []

    faq_text = """
    Course Policies - EdTech Platform

    1. Refund Policy:
    Students can request a full refund within 7 days of enrollment.
    After 7 days, no refunds are allowed under any circumstances.

    2. Assignment Deadlines:
    Each course has weekly assignments.
    Assignments must be submitted before 11:59 PM on the due date.
    Late submissions are not accepted.

    3. Course Access:
    Students will have access to course content for 6 months from enrollment.

    4. Certificate Policy:
    Certificates are issued only after completing 100% of the course and passing all assignments.
    """

    lecture_text = """
    Artificial Intelligence (AI) is the simulation of human intelligence in machines.

    Types of AI:
    - Narrow AI
    - General AI

    Machine Learning is a subset of AI.
    """

    assignment_text = """
    Course: AI
    Assignment 1 Deadline: March 30

    Course: ML
    Assignment 1 Deadline: April 5
    """

    docs.append(Document(page_content=faq_text))
    docs.append(Document(page_content=lecture_text))
    docs.append(Document(page_content=assignment_text))

    return docs


# TOOL DATABASE
student_db = {
    "101": {"name": "Rahul", "course": "AI", "progress": "75%", "status": "active"},
    "102": {"name": "Priya", "course": "ML", "progress": "40%", "status": "inactive"},
    "103": {"name": "Amit", "course": "AI", "progress": "100%", "status": "completed"}
}

def get_student_status(student_id):
    return student_db.get(student_id, "Student not found")

def get_assignment_deadline(course):
    deadlines = {
        "AI": "March 30",
        "ML": "April 5"
    }
    return deadlines.get(course, "Unknown")


# CHUNKING
def split_documents(docs):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=700,
        chunk_overlap=100
    )
    return splitter.split_documents(docs)

# VECTOR STORE
def create_vector_store(chunks):
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    return FAISS.from_documents(chunks, embeddings)


# RETRIEVAL
def retrieve_context(vectordb, query):
    docs = vectordb.similarity_search_with_score(query, k=4)
    return [doc for doc, score in docs]


# AGENT DECISION
def agent_decision(query):
    query = query.lower()

    if "status" in query or "enrollment" in query:
        return "student_tool"
    elif "deadline" in query:
        return "deadline_tool"
    else:
        return "rag"


# PROMPT
def build_prompt(context, query):
    context_text = "\n\n".join([doc.page_content for doc in context])

    return f"""
You are an educational assistant.

Rules:
- Answer ONLY from the context
- If not found, say "I don't know"

Context:
{context_text}

Question: {query}
"""


# BUILD VECTOR DB (ONCE)
documents = load_documents()
chunks = split_documents(documents)
vectordb = create_vector_store(chunks)


# MAIN FUNCTION
def answer_query(query, student_id=None):
    decision = agent_decision(query)

    if decision == "student_tool":
        return f"Student Info: {get_student_status(student_id)}"

    elif decision == "deadline_tool":
        query_lower = query.lower()
        if "ml" in query_lower:
            course = "ML"
        else:
            course = "AI"
        return f"Deadline: {get_assignment_deadline(course)}"

    else:
        context = retrieve_context(vectordb, query)

        if not context:
            return "I couldn't find reliable information."

        prompt = build_prompt(context, query)
        response = llm.invoke(prompt)
        return response.content


# TESTING
if __name__ == "__main__":
    print(answer_query("What is the refund policy?"))
    print(answer_query("What is my enrollment status?", student_id="101"))
    print(answer_query("What is the assignment deadline?"))
    print(answer_query("What is the ML assignment deadline?"))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Students can request a full refund within 7 days of enrollment. After 7 days, no refunds are allowed under any circumstances.
Student Info: {'name': 'Rahul', 'course': 'AI', 'progress': '75%', 'status': 'active'}
Deadline: March 30
Deadline: April 5


In [ ]:
import nbformat
import os

# Get the current notebook's filename
notebook_name = "Que1End_to_end_rag&agent_system.ipynb"
fixed_notebook_name = "Que1End_to_end_rag&agent_system_fixed.ipynb"

if not os.path.exists(notebook_name):
    print(f"Error: The notebook file '{notebook_name}' was not found.")
    print("Please upload the notebook file you wish to fix to your environment or provide the correct path.")
else:
    with open(notebook_name, "r", encoding="utf-8") as f:
        nb = nbformat.read(f, as_version=4)

    # Remove widget metadata if present
    if "widgets" in nb.get("metadata", {}):
        del nb["metadata"]["widgets"]

    with open(fixed_notebook_name, "w", encoding="utf-8") as f:
        nbformat.write(nb, f)

    print(f"Fixed notebook saved to '{fixed_notebook_name}'!")